In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import  roc_auc_score, confusion_matrix, classification_report, roc_curve, auc
import joblib
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
import seaborn as sns
import matplotlib.pyplot as plt
from geopy.distance import geodesic
import matplotlib
matplotlib.use('WebAgg') 
pd.set_option('display.max_rows', None, 'display.max_columns', None)


In [3]:
df = pd.read_csv(Path('data/dataset.csv'))
df = df.loc[:, ~df.columns.str.contains('Unnamed')]
df.shape

(1296675, 22)

In [4]:
df.columns

Index(['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt',
       'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat',
       'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat',
       'merch_long', 'is_fraud'],
      dtype='object')

In [9]:

df[df['is_fraud'] == 0].head()

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [5]:
df['lat'].max()

np.float64(66.6933)

In [6]:
df.describe()

,cc_num,amt,zip,lat,long,city_pop,unix_time,merch_lat,merch_long,is_fraud
count,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06
mean,4.171920e+17,7.035104e+01,4.880067e+04,3.853762e+01,-9.022634e+01,8.882444e+04,1.349244e+09,3.853734e+01,-9.022646e+01,5.788652e-03
std,1.308806e+18,1.603160e+02,2.689322e+04,5.075808e+00,1.375908e+01,3.019564e+05,1.284128e+07,5.109788e+00,1.377109e+01,7.586269e-02
min,6.041621e+10,1.000000e+00,1.257000e+03,2.002710e+01,-1.656723e+02,2.300000e+01,1.325376e+09,1.902779e+01,-1.666712e+02,0.000000e+00
25%,1.800429e+14,9.650000e+00,2.623700e+04,3.462050e+01,-9.679800e+01,7.430000e+02,1.338751e+09,3.473357e+01,-9.689728e+01,0.000000e+00
50%,3.521417e+15,4.752000e+01,4.817400e+04,3.935430e+01,-8.747690e+01,2.456000e+03,1.349250e+09,3.936568e+01,-8.743839e+01,0.000000e+00
75%,4.642255e+15,8.314000e+01,7.204200e+04,4.194040e+01,-8.015800e+01,2.032800e+04,1.359385e+09,4.195716e+01,-8.023680e+01,0.000000e+00
max,4.992346e+18,2.894890e+04,9.978300e+04,6.669330e+01,-6.795030e+01,2.906700e+06,1.371817e+09,6.751027e+01,-6.695090e+01,1.000000e+00


In [7]:
categorical_variable = ['merchant', 'category', 'gender']

encoders = {}
for col in categorical_variable:
    encoders[col] = LabelEncoder()
    df[col] = encoders[col].fit_transform(df[col])
# encoders

In [8]:
df.columns

Index(['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt',
       'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat',
       'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat',
       'merch_long', 'is_fraud'],
      dtype='object')

In [9]:
def haversine_distance(lat1, lon1, lat2, lon2):
    return np.array(geodesic((a, b),(c, d)).km for a, b,c,d in zip(lat1, lon1, lat2, lon2))

df['distance'] = haversine_distance(df['lat'], df['long'], df['merch_lat'], df['merch_long'] )

In [10]:
df['distance'] = df['distance'].astype(float)
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['hours'] = df['trans_date_trans_time'].dt.hour
df['day'] = df['trans_date_trans_time'].dt.day
df['month'] = df['trans_date_trans_time'].dt.month

In [11]:
features = ['hours', 'month', 'day', 'cc_num', 'merchant', 'category', 'amt', 'gender', 'distance']
x= df[features]
y = df['is_fraud']

In [12]:
plt.figure(figsize=(6,4))
sns.countplot(x ='is_fraud', data=df)
plt.title('Class distribution before SMOTE')
plt.show()

In [13]:
smote = SMOTE(random_state=42)
x_resample, y_resample = smote.fit_resample(x, y)

In [14]:
x_resample.shape, y_resample.shape

((2578338, 9), (2578338,))

In [15]:
plt.figure(figsize=(6,4))
sns.countplot(x=y_resample)
plt.title('Class distribution after SMOTE')
plt.show()

In [16]:
x_train, x_test, y_train, y_test = train_test_split(x_resample, y_resample,test_size=0.3 , random_state=42)

In [17]:
## defining the LightGBM Classifier

lgb_model = lgb.LGBMClassifier(max_depth=-1, # no limit of tree depth
                               boosting_type='gbdt', # gradient boosting decision tree
                               objective='binary', # binary classification
                               is_unbalance = True,
                               metrics ='auc', # good for imbalanace dataset
                               learning_rate=0.05, # learning rate in each epoch
                               num_leaves=30, # number of leaves in each trees
                               n_estimators=200 # max number of trees
                               )
lgb_model.fit(x_train, y_train)

[LightGBM] [Info] Number of positive: 902001, number of negative: 902835
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.021735 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1104
[LightGBM] [Info] Number of data points in the train set: 1804836, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499769 -> initscore=-0.000924
[LightGBM] [Info] Start training from score -0.000924


,boosting_type,'gbdt'
,num_leaves,30
,max_depth,-1
,learning_rate,0.05
,n_estimators,200
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [18]:
y_pred = lgb_model.predict(x_test)


In [19]:
print(f'classification_report: \n{classification_report(y_test, y_pred)}')
print(f'roc_auc_score: \n{roc_auc_score(y_test, y_pred)}')


classification_report: 
              precision    recall  f1-score   support

           0       0.92      0.97      0.95    386334
           1       0.97      0.92      0.94    387168

    accuracy                           0.94    773502
   macro avg       0.94      0.94      0.94    773502
weighted avg       0.94      0.94      0.94    773502

roc_auc_score: 
0.9439271440076138


In [21]:
lgb.plot_importance(lgb_model, max_num_features=12, importance_type='split', figsize=(15, 5))
plt.title('Top 10 features with importance metrics')
plt.show()

In [24]:
lgb_model.predict_proba(x_test)

array([[0.97849302, 0.02150698],
       [0.90343075, 0.09656925],
       [0.0021391 , 0.9978609 ],
       ...,
       [0.0107036 , 0.9892964 ],
       [0.99812635, 0.00187365],
       [0.99768341, 0.00231659]], shape=(773502, 2))

In [25]:
fpr, tpr, thresholds = roc_curve(y_test, lgb_model.predict_proba(x_test)[:,1])
roc_auc = auc(fpr, tpr)
fpr, tpr, thresholds, roc_auc

(array([0.        , 0.        , 0.        , ..., 0.99941243, 0.9994176 ,
        1.        ], shape=(158737,)),
 array([0.00000000e+00, 2.58285809e-06, 1.03314323e-05, ...,
        1.00000000e+00, 1.00000000e+00, 1.00000000e+00], shape=(158737,)),
 array([           inf, 9.99378455e-01, 9.99187122e-01, ...,
        7.50290998e-04, 7.49955889e-04, 3.83469867e-04], shape=(158737,)),
 0.9882248377139053)

In [29]:
plt.figure(figsize=(15, 5))
plt.plot(fpr, tpr, color = 'blue', lw = 2, label = f'ROC AUC for (AUC: {roc_auc:.2f})')
plt.plot([0,1],[0,1], color = 'grey', linestyle = '--')
plt.title('ROC')
plt.xlabel('False Positive rate')
plt.ylabel('True positive rate')
plt.legend(loc= "lower right")
plt.show()

In [30]:
joblib.dump(lgb_model, Path('model/CC_fraud_model.jb'))
joblib.dump(encoders, Path('model/encoders.jb'))

['model\\encoders.jb']